In [1]:
!git clone https://github.com/bhujbalatharva252-sketch/Recommendation-Systems-benchmarking.git

Cloning into 'Recommendation-Systems-benchmarking'...
remote: Enumerating objects: 56, done.
remote: Total 56 (delta 0), reused 0 (delta 0), pack-reused 56 (from 1)
Receiving objects: 100% (56/56), 41.38 MiB | 16.61 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Updating files: 100% (31/31), done.


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
%cd Recommendation-Systems-benchmarking

/content/Recommendation-Systems-benchmarking


# LOAD MOVIES

In [5]:

movies = pd.read_csv(
    "data/raw/movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1"
)
movies.head()


,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
movies.columns

Index(['MovieID', 'Title', 'Genres'], dtype='object')

# LOAD RATINGS

In [16]:
train = pd.read_csv("data/processed/train.csv")
train.head()

,UserID,MovieID,Rating,Timestamp,Gender,Age,Occupation,ZipCode,Title,Genres
0,1,3186,4,978300019,F,1,10,48067,"Girl, Interrupted (1999)",Drama
1,1,1270,5,978300055,F,1,10,48067,Back to the Future (1985),Comedy|Sci-Fi
2,1,1721,4,978300055,F,1,10,48067,Titanic (1997),Drama|Romance
3,1,1022,5,978300055,F,1,10,48067,Cinderella (1950),Animation|Children's|Musical
4,1,2340,3,978300103,F,1,10,48067,Meet Joe Black (1998),Romance


# TF-IDF


In [9]:
# Clean Genres
movies["Genres"] = movies["Genres"].fillna("")

In [10]:
# TF-IDF on genres
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["Genres"])

In [11]:
# Cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [12]:
# Movie index mapping
movie_index = pd.Series(movies.index, index=movies["MovieID"])

# Build user profile

In [13]:
def build_user_profile(user_id,train):
    user_movies = train[train["UserID"] == user_id]["MovieID"]        # Get all movies watched by user

    if user_movies.empty:           # if user has no movies
        return None

    idx = []                          # store indices of the watched movies
    for m in user_movies:
        if m in movie_index:
            idx.append(movie_index[m])

    if len(idx) == 0:           # if no movie indices return none
        return None

    return cosine_sim[idx].mean(axis=0)

# Recommendation function

In [14]:
def recommend(user_id, train, top_n=10):
    profile= build_user_profile(user_id, train)

    if profile is None:
        return "No data for user"

    ranked_idx =profile.argsort()[::-1][:top_n]

    return movies.iloc[ranked_idx][["MovieID", "Title", "Genres"]]

In [27]:
# Example
recommend(1,train)

,MovieID,Title,Genres
2033,2102,Steamboat Willie (1940),Animation|Children's|Comedy|Musical
2009,2078,"Jungle Book, The (1967)",Animation|Children's|Comedy|Musical
584,588,Aladdin (1992),Animation|Children's|Comedy|Musical
2011,2080,Lady and the Tramp (1955),Animation|Children's|Comedy|Musical|Romance
2012,2081,"Little Mermaid, The (1989)",Animation|Children's|Comedy|Musical|Romance
1526,1566,Hercules (1997),Adventure|Animation|Children's|Comedy|Musical
360,364,"Lion King, The (1994)",Animation|Children's|Musical
3090,3159,Fantasia 2000 (1999),Animation|Children's|Musical
1019,1032,Alice in Wonderland (1951),Animation|Children's|Musical
1459,1489,Cats Don't Dance (1997),Animation|Children's|Musical
